# 20 — Neural Network Discrete-Time Survival Model with APC Decomposition

**Wang et al. (2024)**, extended to competing risks (prepayment + default).

**Two-stage approach:**
1. **NN-DTSM**: Train one separate MLP per vintage quarter → monthly transition probabilities
2. **APC Decomposition**: Ridge regression on the Lexis graph → Age, Vintage, Calendar-time effects

**Key extension**: Calendar-time effects fitted against macroeconomic variables, then projected forward via AR models for out-of-sample prediction.

**Reference**: Wang, H., Bellotti, A., Qu, R. & Bai, R. (2024). "Discrete-Time Survival Models with Neural Networks for Age–Period–Cohort Analysis of Credit Risk." *Risks*, 12(2), 31.

---

## 1. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

# Project paths
NOTEBOOK_DIR = Path('.').resolve()
PROJECT_DIR = NOTEBOOK_DIR.parent
sys.path.insert(0, str(PROJECT_DIR / 'src'))

DATA_DIR = PROJECT_DIR / 'data' / 'processed'
MODELS_DIR = PROJECT_DIR / 'models'
FIGURES_DIR = PROJECT_DIR / 'reports' / 'figures'

MODELS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

# Imports from project
from competing_risks.nn_dtsm import (
    VintageNNDTSM,
    APCDecomposition,
    prepare_panel_features,
    extract_vintage_quarter,
    get_device,
    STATIC_FEATURES,
    BEHAVIORAL_FEATURES,
    MACRO_FEATURES,
    DEFAULT_INPUT_FEATURES,
    TRAIN_FOLDS,
    VAL_FOLDS,
    TEST_FOLD,
)
from competing_risks.evaluation import (
    time_dependent_concordance_index,
    brier_score_competing_risks,
    EVAL_TIMES,
)

# Plot style
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

# Device
device = get_device('auto')
print(f"PyTorch {torch.__version__}")
print(f"Device:  {device}")
print(f"Eval times: {EVAL_TIMES}")

---

## 2. Data

In [ ]:
# Load loan-month panel
panel = pd.read_parquet(DATA_DIR / 'loan_month_panel.parquet')
panel = prepare_panel_features(panel)

print(f"Panel: {panel.shape[0]:,} rows, {panel['loan_sequence_number'].nunique():,} loans")
print(f"Vintage quarters: {sorted(panel['vintage_quarter'].dropna().unique())}")
print(f"Vintage years: {sorted(panel['vintage_year'].unique())}")
print(f"\nEvent distribution:")
print(panel.groupby('loan_sequence_number')['event_code'].last().value_counts().sort_index().rename(
    {0: 'Censored', 1: 'Prepay', 2: 'Default'}))

# Available features
input_features = [c for c in DEFAULT_INPUT_FEATURES if c in panel.columns]
print(f"\nInput features ({len(input_features)}): {input_features}")
print(f"Macro features: {[c for c in MACRO_FEATURES if c in panel.columns]}")

In [ ]:
# Train / test split (same folds as all other notebooks)
train_panel = panel[panel['fold'].isin(TRAIN_FOLDS)].copy()
test_panel = panel[panel['fold'] == TEST_FOLD].copy()

print(f"Train: {train_panel['loan_sequence_number'].nunique():,} loans, "
      f"{train_panel.shape[0]:,} rows")
print(f"Test:  {test_panel['loan_sequence_number'].nunique():,} loans, "
      f"{test_panel.shape[0]:,} rows")

# Event distribution per split
for name, df in [('Train', train_panel), ('Test', test_panel)]:
    evt = df.groupby('loan_sequence_number')['event_code'].last().value_counts().sort_index()
    print(f"\n{name}: {dict(evt)}")

---

## 3. EDA: Loans per Vintage and Event Rates

In [ ]:
# Loans and events per vintage quarter
loan_level = train_panel.groupby('loan_sequence_number').agg(
    vintage_quarter=('vintage_quarter', 'first'),
    event_code=('event_code', 'last'),
).reset_index()

vq_stats = loan_level.groupby('vintage_quarter').agg(
    n_loans=('loan_sequence_number', 'count'),
    n_prepay=('event_code', lambda x: (x == 1).sum()),
    n_default=('event_code', lambda x: (x == 2).sum()),
).reset_index()
vq_stats['prepay_rate'] = vq_stats['n_prepay'] / vq_stats['n_loans']
vq_stats['default_rate'] = vq_stats['n_default'] / vq_stats['n_loans']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loans per vintage
axes[0].bar(range(len(vq_stats)), vq_stats['n_loans'], color='steelblue', alpha=0.7)
axes[0].set_xticks(range(0, len(vq_stats), max(1, len(vq_stats) // 10)))
axes[0].set_xticklabels(vq_stats['vintage_quarter'].iloc[::max(1, len(vq_stats) // 10)],
                         rotation=45, ha='right')
axes[0].set_ylabel('Number of Loans')
axes[0].set_title('Loans per Vintage Quarter')

# Prepay rate
axes[1].plot(range(len(vq_stats)), vq_stats['prepay_rate'] * 100, 'g-o', markersize=3)
axes[1].set_xticks(range(0, len(vq_stats), max(1, len(vq_stats) // 10)))
axes[1].set_xticklabels(vq_stats['vintage_quarter'].iloc[::max(1, len(vq_stats) // 10)],
                         rotation=45, ha='right')
axes[1].set_ylabel('Prepayment Rate (%)')
axes[1].set_title('Prepayment Rate by Vintage')

# Default rate
axes[2].plot(range(len(vq_stats)), vq_stats['default_rate'] * 100, 'r-o', markersize=3)
axes[2].set_xticks(range(0, len(vq_stats), max(1, len(vq_stats) // 10)))
axes[2].set_xticklabels(vq_stats['vintage_quarter'].iloc[::max(1, len(vq_stats) // 10)],
                         rotation=45, ha='right')
axes[2].set_ylabel('Default Rate (%)')
axes[2].set_title('Default Rate by Vintage')

plt.tight_layout()
plt.show()

print(vq_stats.to_string(index=False))

---

## 4. NN-DTSM Training

Train one separate MLP per vintage quarter. Each subnetwork has a 3-class softmax output (current, prepay, default) with per-vintage class-weighted cross-entropy loss.

In [ ]:
%%time

# Model configuration (matching Wang et al. grid search optimum)
model = VintageNNDTSM(
    n_hidden_layers=4,
    n_neurons=8,
    dropout=0.0,
    n_epochs=30,
    batch_size=256,
    learning_rate=1e-3,
    weight_decay=1e-4,
    patience=10,
    use_class_weights=True,
    device=str(device),
    random_seed=42,
)

model.fit(train_panel, input_features=input_features)

In [ ]:
# Training curves: show a few representative vintages
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
sample_vintages = sorted(model.subnets_.keys())[:6]

for ax, vq in zip(axes.flat, sample_vintages):
    hist = model.histories_[vq]
    ax.plot(hist['train_loss'], label='Train', alpha=0.8)
    ax.plot(hist['val_loss'], label='Val', alpha=0.8)
    ax.set_title(f'{vq} ({len(hist["val_loss"])} epochs)')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=8)

plt.suptitle('Training Curves (sample vintages)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Save model
model.save(str(MODELS_DIR / 'nn_dtsm_model.pt'))
print(f"Saved model with {len(model.subnets_)} subnetworks")

---

## 5. NN-DTSM vs Linear DTSM: McFadden Pseudo-R²

Compare the neural network's fit against a null (intercept-only) model per vintage. Positive R² indicates the NN captures covariate effects beyond class frequencies.

In [ ]:
# McFadden pseudo-R² on test set per vintage
r2_dict = model.mcfadden_pseudo_r2(test_panel)
r2_df = pd.DataFrame([
    {'vintage_quarter': vq, 'pseudo_r2': r2}
    for vq, r2 in sorted(r2_dict.items())
])

fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(range(len(r2_df)), r2_df['pseudo_r2'], color='steelblue', alpha=0.7)
ax.set_xticks(range(0, len(r2_df), max(1, len(r2_df) // 15)))
ax.set_xticklabels(r2_df['vintage_quarter'].iloc[::max(1, len(r2_df) // 15)],
                   rotation=45, ha='right')
ax.axhline(0, color='red', linestyle='--', alpha=0.5)
ax.set_ylabel('McFadden Pseudo-R²')
ax.set_title('NN-DTSM: McFadden Pseudo-R² per Vintage (Test Set)')
plt.tight_layout()
plt.show()

print(f"Mean R²:  {r2_df['pseudo_r2'].mean():.4f}")
print(f"Median:   {r2_df['pseudo_r2'].median():.4f}")
print(f"Range:    [{r2_df['pseudo_r2'].min():.4f}, {r2_df['pseudo_r2'].max():.4f}]")

---

## 6. CIF Prediction

Compute cumulative incidence functions by chaining monthly transition probabilities on the test set.

In [ ]:
%%time

cif_result = model.predict_cif(test_panel, eval_times=EVAL_TIMES)
print(f"Test loans with CIF: {len(cif_result['loan_sequence_number']):,}")

for t in EVAL_TIMES:
    cif_p = cif_result[f'cif_prepay_{t}']
    cif_d = cif_result[f'cif_default_{t}']
    print(f"  t={t}: CIF_prepay={cif_p.mean():.4f} (std={cif_p.std():.4f}), "
          f"CIF_default={cif_d.mean():.4f} (std={cif_d.std():.4f}), "
          f"Total={(cif_p + cif_d).mean():.4f}")

---

## 7. Evaluation: C-index and Brier Score

In [ ]:
event_times = cif_result['duration']
event_codes = cif_result['event_code']

eval_rows = []
for t in EVAL_TIMES:
    c_prepay, _, _ = time_dependent_concordance_index(
        event_times, event_codes,
        cif_result[f'cif_prepay_{t}'], eval_time=t, event_of_interest=1)

    c_default, _, _ = time_dependent_concordance_index(
        event_times, event_codes,
        cif_result[f'cif_default_{t}'], eval_time=t, event_of_interest=2)

    bs_prepay = brier_score_competing_risks(
        event_times, event_codes,
        cif_result[f'cif_prepay_{t}'], eval_time=t, event_of_interest=1)

    bs_default = brier_score_competing_risks(
        event_times, event_codes,
        cif_result[f'cif_default_{t}'], eval_time=t, event_of_interest=2)

    eval_rows.append({
        'Horizon': t,
        'C_prepay': c_prepay, 'C_default': c_default,
        'BS_prepay': bs_prepay, 'BS_default': bs_default,
    })

eval_df = pd.DataFrame(eval_rows)
print("NN-DTSM + APC — Time-Dependent Evaluation (Test Set)")
print("=" * 60)
print(eval_df.to_string(index=False, float_format='%.4f'))

# Save
eval_df.to_csv(MODELS_DIR / 'nn_dtsm_cindex.csv', index=False)

---

## 8. Lexis Graphs

Heatmaps of predicted hazard rates across Age (loan maturity) x Vintage (origination quarter), separately for prepayment and default.

In [ ]:
%%time

# Build Lexis data from training predictions
lexis = model.build_lexis_data(train_panel)

for cause, cause_name in [(1, 'Prepayment'), (2, 'Default')]:
    lexis_df = lexis[cause]
    print(f"{cause_name}: {len(lexis_df)} Lexis cells")

    pivot = lexis_df.pivot_table(
        values='mean_prob', index='loan_age',
        columns='vintage_quarter', aggfunc='mean')

    fig, ax = plt.subplots(figsize=(14, 8))
    im = ax.pcolormesh(
        range(pivot.shape[1]), pivot.index, pivot.values,
        shading='auto', cmap='YlOrRd')
    plt.colorbar(im, ax=ax, label=f'P({cause_name})')

    n_cols = pivot.shape[1]
    step = max(1, n_cols // 15)
    ax.set_xticks(range(0, n_cols, step))
    ax.set_xticklabels([str(pivot.columns[i]) for i in range(0, n_cols, step)],
                       rotation=45, ha='right')
    ax.set_xlabel('Vintage Quarter')
    ax.set_ylabel('Loan Age (months)')
    ax.set_title(f'Lexis Graph: Monthly {cause_name} Hazard Rate')
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / f'nn_dtsm_lexis_{cause_name.lower()}.png',
                dpi=150, bbox_inches='tight')
    plt.show()

---

## 9. APC Decomposition (Prepayment)

Ridge regression decomposes the Lexis graph into Age, Vintage, and Calendar-time effects.

In [ ]:
# APC decomposition — Prepayment
apc_prepay = APCDecomposition()
apc_prepay.fit(lexis[1])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
apc_prepay.plot_effects(title_prefix='Prepay', axes=axes)
plt.suptitle('APC Decomposition — Prepayment', fontsize=14, y=1.02)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'nn_dtsm_apc_prepay.png', dpi=150, bbox_inches='tight')
plt.show()

effects_p = apc_prepay.get_effects()
for name, eff in effects_p.items():
    print(f"  {name}: [{eff.min():.6f}, {eff.max():.6f}], "
          f"range={eff.max() - eff.min():.6f}")

---

## 10. APC Decomposition (Default)

In [ ]:
# APC decomposition — Default
apc_default = APCDecomposition()

if len(lexis[2]) >= 10:
    apc_default.fit(lexis[2])

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    apc_default.plot_effects(title_prefix='Default', axes=axes)
    plt.suptitle('APC Decomposition — Default', fontsize=14, y=1.02)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / 'nn_dtsm_apc_default.png', dpi=150, bbox_inches='tight')
    plt.show()

    effects_d = apc_default.get_effects()
    for name, eff in effects_d.items():
        print(f"  {name}: [{eff.min():.6f}, {eff.max():.6f}], "
              f"range={eff.max() - eff.min():.6f}")
else:
    print(f"Too few default Lexis cells ({len(lexis[2])}) for APC decomposition. "
          f"Use the full vintage dataset for meaningful default APC.")

---

## 11. Calendar-Time vs Macro Regression

Fit the estimated calendar-time coefficients against macroeconomic variables to:
1. Solve the APC identification problem
2. Provide economic interpretability
3. Enable forward projection via AR models

In [ ]:
macro_cols = [c for c in MACRO_FEATURES if c in train_panel.columns]
print(f"Available macro variables: {macro_cols}")

# Prepayment: calendar-time macro regression
print("\n--- Prepayment ---")
try:
    apc_prepay.fit_macro_regression(train_panel, macro_cols=macro_cols)

    reg = apc_prepay.macro_regression_
    print(f"\nR² = {reg.rsquared:.4f}, Adj R² = {reg.rsquared_adj:.4f}")
    print(f"\nCoefficients:")
    for name, coef, pval in zip(
        ['const'] + macro_cols + ['trend'],
        reg.params, reg.pvalues
    ):
        sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else ''
        print(f"  {name:<20} {coef:>10.6f}  p={pval:.4f} {sig}")
except Exception as e:
    print(f"  Failed: {e}")

# Default: calendar-time macro regression (if APC was fitted)
if apc_default.calendar_effects_ is not None:
    print("\n--- Default ---")
    try:
        apc_default.fit_macro_regression(train_panel, macro_cols=macro_cols)
        reg_d = apc_default.macro_regression_
        print(f"\nR² = {reg_d.rsquared:.4f}, Adj R² = {reg_d.rsquared_adj:.4f}")
    except Exception as e:
        print(f"  Failed: {e}")

---

## 12. AR Macro Projection

Fit AR(p) models on each macro variable and project the calendar-time effect forward via Monte Carlo simulation. This produces out-of-sample predictions with uncertainty bands.

In [ ]:
# Fit AR models and project forward — Prepayment
if apc_prepay.macro_regression_ is not None:
    print("--- Prepayment: AR Macro Projection ---")
    apc_prepay.fit_ar_models(train_panel, macro_cols=macro_cols, max_lag=4)

    proj_prepay = apc_prepay.project_calendar_effect(
        n_months_ahead=max(EVAL_TIMES), n_mc_paths=500, seed=42)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(proj_prepay['month_ahead'], proj_prepay['gamma_mean'],
            'b-', linewidth=2, label='Mean projection')
    ax.fill_between(proj_prepay['month_ahead'],
                    proj_prepay['gamma_lo'], proj_prepay['gamma_hi'],
                    alpha=0.3, color='steelblue', label='90% CI')
    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Months Ahead')
    ax.set_ylabel('Projected γ(c*)')
    ax.set_title('Prepayment: AR-Projected Calendar-Time Effect')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / 'nn_dtsm_ar_proj_prepay.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\nProjection uncertainty (std) at key horizons:")
    for t in EVAL_TIMES:
        row = proj_prepay[proj_prepay['month_ahead'] == t]
        if len(row) > 0:
            print(f"  t={t}: mean={row['gamma_mean'].values[0]:.6f}, "
                  f"std={row['gamma_std'].values[0]:.6f}")
else:
    print("Macro regression not fitted — skipping AR projection")

In [ ]:
# AR projection — Default (if APC was fitted)
if apc_default.macro_regression_ is not None:
    print("--- Default: AR Macro Projection ---")
    apc_default.fit_ar_models(train_panel, macro_cols=macro_cols, max_lag=4)

    proj_default = apc_default.project_calendar_effect(
        n_months_ahead=max(EVAL_TIMES), n_mc_paths=500, seed=42)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(proj_default['month_ahead'], proj_default['gamma_mean'],
            'r-', linewidth=2, label='Mean projection')
    ax.fill_between(proj_default['month_ahead'],
                    proj_default['gamma_lo'], proj_default['gamma_hi'],
                    alpha=0.3, color='salmon', label='90% CI')
    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Months Ahead')
    ax.set_ylabel('Projected γ(c*)')
    ax.set_title('Default: AR-Projected Calendar-Time Effect')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / 'nn_dtsm_ar_proj_default.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Default APC not fitted — skipping")

---

## 13. Out-of-Sample Evaluation

Compare CIF predictions using three calendar-time variants:
- **Oracle**: actual future macro values (upper bound)
- **AR-projected**: AR-simulated macro paths (realistic scenario)
- **Frozen**: last observed calendar-time effect (naive baseline)

In [ ]:
# Out-of-sample evaluation placeholder
# With the sampled panel (post-2010 data, 44 defaults), this analysis
# is limited. On the full vintage dataset (1999-2025, 19,728 defaults),
# the comparison becomes meaningful:
#
# 1. Train on 1999-2017 vintages
# 2. For 2018-2020 test loans:
#    - Oracle: plug actual 2018-2020 macro values into gamma regression
#    - AR-projected: project from 2017 endpoint via AR models
#    - Frozen: use gamma at 2017 endpoint for all future months
# 3. Compare C-index and Brier score across the three variants

print("Out-of-sample evaluation with AR projection requires the full vintage dataset.")
print("Use 'scripts/run_nn_dtsm.py' on the supercomputer for the complete analysis.")
print()
print("The three-variant comparison (oracle / AR / frozen) quantifies:")
print("  - How much the calendar-time macro projection improves over naive freezing")
print("  - How much room remains vs. the oracle (perfect macro knowledge)")

---

## 14. Cross-Cause Comparison

Compare APC effects between prepayment and default to understand how time-related risk drivers differ across the two competing events.

In [ ]:
# Side-by-side APC comparison
if apc_prepay.age_effects_ is not None and apc_default.age_effects_ is not None:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Age effects
    axes[0].plot(apc_prepay.age_effects_.index, apc_prepay.age_effects_.values,
                 'g-', linewidth=2, label='Prepay')
    axes[0].plot(apc_default.age_effects_.index, apc_default.age_effects_.values,
                 'r-', linewidth=2, label='Default')
    axes[0].set_xlabel('Loan Age (months)')
    axes[0].set_ylabel('α(t)')
    axes[0].set_title('Age Effect')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)

    # Vintage effects
    # Align on common vintages
    common_vin = sorted(set(apc_prepay.vintage_effects_.index) &
                         set(apc_default.vintage_effects_.index))
    if common_vin:
        axes[1].bar(np.arange(len(common_vin)) - 0.15,
                    [apc_prepay.vintage_effects_[v] for v in common_vin],
                    width=0.3, color='green', alpha=0.6, label='Prepay')
        axes[1].bar(np.arange(len(common_vin)) + 0.15,
                    [apc_default.vintage_effects_[v] for v in common_vin],
                    width=0.3, color='red', alpha=0.6, label='Default')
        step = max(1, len(common_vin) // 10)
        axes[1].set_xticks(range(0, len(common_vin), step))
        axes[1].set_xticklabels([str(common_vin[i]) for i in range(0, len(common_vin), step)],
                               rotation=45, ha='right')
    axes[1].set_ylabel('β(v)')
    axes[1].set_title('Vintage Effect')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3, axis='y')
    axes[1].axhline(0, color='gray', linestyle='--', alpha=0.5)

    # Calendar effects
    axes[2].plot(range(len(apc_prepay.calendar_effects_)),
                 apc_prepay.calendar_effects_.values,
                 'g-', linewidth=1.5, label='Prepay', alpha=0.8)
    axes[2].plot(range(len(apc_default.calendar_effects_)),
                 apc_default.calendar_effects_.values,
                 'r-', linewidth=1.5, label='Default', alpha=0.8)
    axes[2].set_ylabel('γ(c)')
    axes[2].set_title('Calendar-Time Effect')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    axes[2].axhline(0, color='gray', linestyle='--', alpha=0.5)

    plt.suptitle('APC Cross-Cause Comparison: Prepayment vs Default',
                 fontsize=14, y=1.02)
    plt.tight_layout()
    fig.savefig(FIGURES_DIR / 'nn_dtsm_apc_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Both APC decompositions needed for cross-cause comparison.")

---

## 15. Model Comparison

Combined results table comparing NN-DTSM + APC with all other models in the study.

In [ ]:
# Load previous model results (if available)
comparison_rows = []

# NN-DTSM results from this notebook
for _, row in eval_df.iterrows():
    comparison_rows.append({
        'Model': 'NN-DTSM + APC',
        'Horizon': int(row['Horizon']),
        'C_prepay': row['C_prepay'],
        'C_default': row['C_default'],
        'BS_prepay': row['BS_prepay'],
        'BS_default': row['BS_default'],
    })

# Try to load other model results
other_models = {
    'Sadhwani NN': 'sadhwani_cindex.csv',
    'DeepHit': 'deephit_cindex.csv',
    'Joint Model': 'joint_model_cindex.csv',
}

for model_name, filename in other_models.items():
    fpath = MODELS_DIR / filename
    if fpath.exists():
        try:
            prev = pd.read_csv(fpath)
            for _, row in prev.iterrows():
                comparison_rows.append({
                    'Model': model_name,
                    'Horizon': int(row.get('Horizon', row.get('horizon', 0))),
                    'C_prepay': row.get('C_prepay', row.get('c_prepay', np.nan)),
                    'C_default': row.get('C_default', row.get('c_default', np.nan)),
                    'BS_prepay': row.get('BS_prepay', np.nan),
                    'BS_default': row.get('BS_default', np.nan),
                })
        except Exception:
            pass

if comparison_rows:
    comp_df = pd.DataFrame(comparison_rows)
    print("Model Comparison — Time-Dependent C-index (Test Set)")
    print("=" * 70)

    for t in EVAL_TIMES:
        subset = comp_df[comp_df['Horizon'] == t]
        if len(subset) > 0:
            print(f"\n  Horizon = {t} months")
            print(subset[['Model', 'C_prepay', 'C_default']].to_string(
                index=False, float_format='%.4f'))
else:
    print("No comparison data available — run other notebooks first.")